In [ ]:
#Importing relevent packages
import sys
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature
import matplotlib.pyplot as plt
import numpy as np

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from saws_functions import sarg_grid_from_sat  # noqa: E402

In [ ]:
#Loading satellite images
image_name_Central_Atlantic = "../SaWSdata/C20241772024183.1KM.C_ATLANTIC.7DAY.L3D.FA_UNET_DENSITY.png"
image_name_Central_East_Atlantic = "../SaWSdata/C20241772024183.1KM.CE_ATLANTIC.7DAY.L3D.FA_UNET_DENSITY.png"

#Applying function on images
sarg_lon_grid_C, sarg_lat_grid_C, amount_C = sarg_grid_from_sat(image_name_Central_Atlantic, 22.0, 0.0, -38.0, -63.0, coarse=True)
sarg_lon_grid_CE, sarg_lat_grid_CE, amount_CE = sarg_grid_from_sat(image_name_Central_East_Atlantic, 22.0, 0.0, -11.5, -38.0, coarse=True)

#Adding the grids of both images to 1 longitude array and 1 latitude array and 1 amount
sarg_LONGITUDES = np.append(sarg_lon_grid_C, sarg_lon_grid_CE, axis=0)
sarg_LATITUDES  = np.append(sarg_lat_grid_C, sarg_lat_grid_CE, axis=0)
sarg_AMOUNT = amount_C + amount_CE

#Creating figure layout: top row = 2 PNG images, bottom row = map
fig = plt.figure(figsize=(12, 8), constrained_layout=True)
gs = fig.add_gridspec(2, 2)

#Top-left: Central Atlantic PNG
ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(plt.imread(image_name_Central_Atlantic))
ax1.axis('off')
ax1.set_anchor('E')
ax1.set_title('a) SaWS image of Central Atlantic on 1 July 2024')

#Top-right: Central-East Atlantic PNG
ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(plt.imread(image_name_Central_East_Atlantic))
ax2.axis('off')
ax2.set_anchor('W')
ax2.set_title('b) SaWS image of Central-East Atlantic on 1 July 2024')

#Bottom row: map spanning both columns
ax3 = fig.add_subplot(gs[1, :], projection=ccrs.PlateCarree())
ax3.add_feature(cartopy.feature.COASTLINE.with_scale('10m'), zorder=2)
ax3.add_feature(cartopy.feature.LAND.with_scale('10m'), zorder=1)
ax3.gridlines(draw_labels=['left', 'bottom'], zorder=0, alpha=0.3, linestyle='--')
ax3.scatter(sarg_LONGITUDES, sarg_LATITUDES, s=0.1, color='olivedrab', zorder=5)
ax3.set_extent([-75, -12, -2, 21])
ax3.set_title(f'c) Satellite-based release locations of {sarg_AMOUNT:,} virtual particles on 1 July 2024')

fig.savefig('Figure1.pdf', bbox_inches='tight', dpi=300)

plt.show()